In [1]:
import pandas as pd

# Possible values for each attribute
attribute_values = {
    'Sky': ['Sunny', 'Cloudy', 'Rainy'],
    'AirTemp': ['Warm', 'Cold'],
    'Humidity': ['Normal', 'High'],
    'Wind': ['Strong', 'Weak'],
    'Water': ['Warm', 'Cool'],
    'Forecast': ['Same', 'Change']
}

attributes_map = {
    0: 'Sky',
    1: 'AirTemp',
    2: 'Humidity',
    3: 'Wind',
    4: 'Water',
    5: 'Forecast'
}

In [2]:
# Check whether h1 is more general than h2
def more_general(h1, h2):
    for x, y in zip(h1, h2):
        if x != '?' and x != y and y != '0':
            return False
    return True


# Check whether a hypothesis covers an example
def covers(example, hypothesis):
    return all(
        h == '?' or h == x
        for h, x in zip(hypothesis, example)
    )


# Check consistency
def consistent(h, example, label):
    prediction = covers(example, h)

    if label == 'Yes':
        return prediction
    else:
        return not prediction

In [3]:
# Generate minimal generalizations of S
def min_generalizations(s, example):
    h = list(s)

    for i in range(len(s)):
        if s[i] != example[i]:
            h[i] = '?'

    return [h]


# Generate minimal specializations of G
def min_specializations(g, example, sb):
    specializations = []

    for i in range(len(g)):
        if g[i] == '?':
            for value in attribute_values[attributes_map[i]]:
                if value != example[i]:
                    h = list(g)
                    h[i] = value

                    # Must be more general than at least one S
                    if any(more_general(h, s) for s in sb):
                        specializations.append(h)

    return specializations

In [4]:
# Print S and G boundaries
def print_sg_gb(s, g):
    print("S:")
    for h in s:
        print(h)

    print("G:")
    for h in g:
        print(h)

    print()

In [5]:
# Candidate Elimination Algorithm

def candidate_elimination_algorithm(data):
    n = len(data.columns) - 1

    # Initialize S and G boundaries
    S = [['0'] * n]
    G = [['?'] * n]

    print("Step 0")
    print_sg_gb(S, G)

    for index, row in data.iterrows():
        example = row.iloc[:-1].tolist()
        label = row.iloc[-1]

        # POSITIVE EXAMPLE
        if label == 'Yes':

            # Remove G hypotheses that do not cover example
            G = [
                g for g in G
                if covers(example, g)
            ]

            new_S = []

            for s in S:
                if not covers(example, s):

                    # If S is the initial most-specific hypothesis
                    if all(x == '0' for x in s):
                        new_S.append(example)
                    else:
                        # Generalize S
                        h = min_generalizations(s, example)[0]

                        # Keep only if it is not more general
                        # than something in G
                        if any(more_general(g, h) for g in G):
                            new_S.append(h)
                else:
                    new_S.append(s)

            S = new_S

        # NEGATIVE EXAMPLE
        else:

            # Remove S hypotheses that cover negative example
            S = [
                s for s in S
                if not covers(example, s)
            ]

            new_G = []

            for g in G:
                if covers(example, g):

                    # Specialize G
                    new_G.extend(
                        min_specializations(g, example, S)
                    )
                else:
                    new_G.append(g)

            G = new_G

        print("Step", index + 1)
        print_sg_gb(S, G)

    return S, G

In [6]:
# Training dataset
training_examples = pd.DataFrame([
    ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same', 'Yes'],
    ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change', 'No'],
    ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change', 'Yes']
], columns=[
    'Sky',
    'AirTemp',
    'Humidity',
    'Wind',
    'Water',
    'Forecast',
    'EnjoySport'
])

# Run Candidate Elimination
specific_boundary, general_boundary = \
    candidate_elimination_algorithm(training_examples)

print("FINAL VERSION SPACE")
print_sg_gb(specific_boundary, general_boundary)

Step 0
S:
['0', '0', '0', '0', '0', '0']
G:
['?', '?', '?', '?', '?', '?']

Step 1
S:
['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
G:
['?', '?', '?', '?', '?', '?']

Step 2
S:
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G:
['?', '?', '?', '?', '?', '?']

Step 3
S:
['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
G:
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']
['?', '?', '?', '?', '?', 'Same']

Step 4
S:
['Sunny', 'Warm', '?', 'Strong', '?', '?']
G:
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']

FINAL VERSION SPACE
S:
['Sunny', 'Warm', '?', 'Strong', '?', '?']
G:
['Sunny', '?', '?', '?', '?', '?']
['?', 'Warm', '?', '?', '?', '?']

